# DINO dense degradation sweep: all Drive checkpoints

This Colab notebook evaluates every recognizable DINO ViT-S/16 checkpoint in a Google Drive checkpoint folder. It follows the dense degradation protocol used in *Exploring Structural Degradation in Dense Representations for Self-supervised Learning*: frozen backbone, projector removed, last-layer patch embeddings, lightweight linear semantic segmentation head, PASCAL VOC mIoU curve.

It also runs the patch-level diagnostic suite requested during supervision: DSE class separability/effective rank, patch feature magnitude histograms, CLS-to-patch cosine, CLS attention maps, fixed-query patch similarity maps, and deterministic fixed-basis PCA patch-feature maps.

By default this notebook scans `/content/drive/MyDrive/dinocheckpoint` and runs all `checkpoint*.pth` files it can identify. Update the paths in the configuration cell before running if your Drive layout differs.


## 1. Mount Google Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import sys

WORK_ROOT = Path('/content')
REPO_DIR = WORK_ROOT / 'dino'
BRANCH = 'codex/raw-l2-diagnostics'
REPO_URL = 'https://github.com/xbz123/dino-dense-degradation.git'

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

subprocess.run(['grep', '-R', 'raw_dse', '-n', str(REPO_DIR / 'analyze_patch_statistics.py')], check=True)
subprocess.run(['grep', '-R', 'fig_raw_vs_l2_dse', '-n', str(REPO_DIR / 'plot_dense_diagnostics.py')], check=True)

sys.path.insert(0, str(REPO_DIR))

## 2. Configure paths and evaluation settings

In [ ]:
from pathlib import Path

# Google Drive folder containing checkpoint*.pth files.
# The notebook auto-discovers every recognizable checkpoint in this folder, including names like:
# checkpoint03.pth, checkpoint0020.pth, checkpoint0125.pth, checkpoint0210.pth, checkpoint215.pth.
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/dinocheckpoint')

# Optional filter. Keep None to evaluate every checkpoint found in DRIVE_CHECKPOINT_DIR.
# Example for a quick smoke test: CHECKPOINT_EPOCH_FILTER = [180, 190, 200, 210, 215]
CHECKPOINT_EPOCH_FILTER = None

# ImageNet-style image folder used for DSE/patch diagnostics. This should match the pretraining data distribution.
# It must be readable by torchvision.datasets.ImageFolder: root/class_name/image.jpg.
DSE_IMAGE_ROOT_CANDIDATES = [
    '/content/drive/MyDrive/imagenet100/train',
    '/content/drive/MyDrive/ImageNet100/train',
]
DSE_IMAGE_ROOT = Path(next((p for p in DSE_IMAGE_ROOT_CANDIDATES if Path(p).is_dir()), DSE_IMAGE_ROOT_CANDIDATES[0]))

OUTPUT_ROOT = Path('/content/drive/MyDrive/dino_dense_degradation_eval')
OUTPUT_RUN_SUFFIX = 'raw_l2'
WORK_CKPT_DIR = Path('/content/dino_eval_checkpoints')

# Independent run switches. Turn on only the expensive stages you need.
RUN_PATCH_DIAGNOSTICS = True
RUN_VOC_EVAL = False
RUN_PLOT_REPORT = True
INSPECT_OUTPUTS = True
CREATE_OUTPUT_ARCHIVE = True

# PASCAL VOC frozen-backbone linear segmentation settings.
VOC_ROOT = Path('/content/voc_data')
VOC_IMG_SIZE = 480
VOC_TRAIN_EPOCHS = 15
VOC_LR = 0.01
VOC_BATCH_SIZE = 32
VOC_OPTIMIZER = 'adam'
VOC_PROBE_SEED = 42
VOC_FEATURE_DTYPE = 'float16'

# Patch/DSE diagnostics. Paper-style DSE uses 2048 sampled pretraining images.
# For a smoke test, set NUM_DSE_IMAGES=128 and PATCH_DSE_GROUP_STRIDE=8.
NUM_DSE_IMAGES = 2048
NUM_VIS_IMAGES = 6
PATCH_DIAG_BATCH_SIZE = 32
PATCH_DIAG_NUM_WORKERS = 2
CHECKPOINT_KEY = 'teacher'
MAX_SPECTRUM_TOKENS = 30000
MAX_KMEANS_TOKENS = 12000
PATCH_DSE_GROUP_STRIDE = 1
STRICT_INTERNAL_EPOCH = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_CKPT_DIR.mkdir(parents=True, exist_ok=True)
VOC_ROOT.mkdir(parents=True, exist_ok=True)

print('checkpoint dir:', DRIVE_CHECKPOINT_DIR)
print('checkpoint filter:', CHECKPOINT_EPOCH_FILTER)
print('DSE image root:', DSE_IMAGE_ROOT)
print('base output root:', OUTPUT_ROOT)
print('output suffix:', OUTPUT_RUN_SUFFIX)
print('run patch diagnostics:', RUN_PATCH_DIAGNOSTICS)
print('run VOC eval:', RUN_VOC_EVAL)
print('VOC optimizer:', VOC_OPTIMIZER)
print('VOC probe seed:', VOC_PROBE_SEED)
print('run plot/report:', RUN_PLOT_REPORT)
print('create archive:', CREATE_OUTPUT_ARCHIVE)

## 3. Auto-discover, copy, and verify checkpoints from Google Drive

The evaluator expects epoch numbers in filenames. This cell scans the Drive folder, infers the epoch from names like `checkpoint0120.pth` or `checkpoint215.pth`, optionally uses `checkpoint.pth` if it has an internal epoch and no epoch-named duplicate exists, and normalizes everything to `checkpoint####.pth` under `/content/dino_eval_checkpoints`.


In [ ]:
import json
import os
import shutil
from pathlib import Path
from dense_eval_utils import build_run_output_root, discover_checkpoint_files

assert DRIVE_CHECKPOINT_DIR.is_dir(), DRIVE_CHECKPOINT_DIR
print('=== Drive checkpoint files ===')
all_drive_files = sorted(path.name for path in DRIVE_CHECKPOINT_DIR.iterdir())
print('\n'.join(all_drive_files[:500]))

selected = discover_checkpoint_files(DRIVE_CHECKPOINT_DIR, epoch_filter=CHECKPOINT_EPOCH_FILTER)
assert selected, f'No recognizable checkpoint*.pth files found in {DRIVE_CHECKPOINT_DIR}'

# Clear old normalized checkpoint files from this runtime so repeated runs cannot mix old selections.
for path in WORK_CKPT_DIR.glob('checkpoint*.pth'):
    path.unlink()

prepared = []
print('=== Selected checkpoints ===')
for item in selected:
    dst = WORK_CKPT_DIR / f'checkpoint{item.epoch:04d}.pth'
    shutil.copy2(item.path, dst)
    print(f'{item.epoch:>4}: {item.path.name} -> {dst} | internal_epoch={item.internal_epoch} | {item.size_mb:.1f} MB')
    prepared.append({
        'epoch': item.epoch,
        'source': str(item.path),
        'normalized': str(dst),
        'internal_epoch': item.internal_epoch,
        'size_mb': item.size_mb,
    })

(WORK_CKPT_DIR / 'selected_checkpoints.json').write_text(json.dumps(prepared, indent=2))

SELECTED_EPOCHS = [item['epoch'] for item in prepared]
FINAL_EPOCH = max(SELECTED_EPOCHS)
BASE_RUN_OUTPUT_ROOT = build_run_output_root(OUTPUT_ROOT, SELECTED_EPOCHS)
RUN_OUTPUT_ROOT = build_run_output_root(OUTPUT_ROOT, SELECTED_EPOCHS, suffix=OUTPUT_RUN_SUFFIX)
RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(WORK_CKPT_DIR / 'selected_checkpoints.json', RUN_OUTPUT_ROOT / 'selected_checkpoints.json')

VOC_JSON_CANDIDATES = [
    RUN_OUTPUT_ROOT / 'voc_all_checkpoints' / 'voc_miou_results.json',
    BASE_RUN_OUTPUT_ROOT / 'voc_all_checkpoints' / 'voc_miou_results.json',
]
for root in [OUTPUT_ROOT]:
    if root.is_dir():
        VOC_JSON_CANDIDATES.extend(sorted(root.glob('to_epoch_*/voc_all_checkpoints/voc_miou_results.json'), reverse=True))
VOC_JSON_FOR_REPORT = next((path for path in VOC_JSON_CANDIDATES if path.is_file()), None)

print('=== Prepared epochs ===')
print(SELECTED_EPOCHS)
print('final epoch:', FINAL_EPOCH)
print('base run output root:', BASE_RUN_OUTPUT_ROOT)
print('run output root:', RUN_OUTPUT_ROOT)
print('VOC json for report:', VOC_JSON_FOR_REPORT)
print('=== Prepared files ===')
print('\n'.join(sorted(path.name for path in WORK_CKPT_DIR.iterdir())))

## 4. Optional PASCAL VOC linear segmentation

Set `RUN_VOC_EVAL = True` in section 2 to run the paper-style frozen-backbone PASCAL VOC linear segmentation sweep. If it is `False`, the notebook reuses any existing `voc_miou_results.json` it can find; otherwise the raw/L2 report is generated without VOC.

In [ ]:
import subprocess
import sys

VOC_OUTPUT_DIR = RUN_OUTPUT_ROOT / 'voc_all_checkpoints'

if RUN_VOC_EVAL:
    VOC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    voc_cmd = [
        sys.executable,
        str(REPO_DIR / 'eval_voc_dense.py'),
        '--ckpt_dir', str(WORK_CKPT_DIR),
        '--voc_root', str(VOC_ROOT),
        '--arch', 'vit_small',
        '--patch_size', '16',
        '--img_size', str(VOC_IMG_SIZE),
        '--train_epochs', str(VOC_TRAIN_EPOCHS),
        '--lr', str(VOC_LR),
        '--batch_size', str(VOC_BATCH_SIZE),
        '--optimizer', VOC_OPTIMIZER,
        '--probe_seed', str(VOC_PROBE_SEED),
        '--feature_dtype', VOC_FEATURE_DTYPE,
        '--output_dir', str(VOC_OUTPUT_DIR),
    ]
    print(' '.join(voc_cmd))
    subprocess.run(voc_cmd, check=True)
    VOC_JSON_FOR_REPORT = VOC_OUTPUT_DIR / 'voc_miou_results.json'
    assert VOC_JSON_FOR_REPORT.is_file(), VOC_JSON_FOR_REPORT
    print('Wrote VOC JSON:', VOC_JSON_FOR_REPORT)
elif VOC_JSON_FOR_REPORT is None:
    print('No VOC JSON found. Continuing with patch-only raw/L2 diagnostics.')
else:
    print('Using VOC JSON:', VOC_JSON_FOR_REPORT)

## 5. Optional raw/L2 patch diagnostics

Set `RUN_PATCH_DIAGNOSTICS = True` to compute DSE, raw/L2 structural metrics, attention/PCA/query visualizations, and per-checkpoint CSV/JSON outputs. For a quick smoke test, set `CHECKPOINT_EPOCH_FILTER = [215]`, reduce `NUM_DSE_IMAGES`, and keep VOC disabled.

In [ ]:
import subprocess
import sys

summary_csv = RUN_OUTPUT_ROOT / 'patch_attention_dse_all_checkpoints' / 'patch_attention_dse_summary.csv'

if RUN_PATCH_DIAGNOSTICS:
    cmd = [
        sys.executable,
        str(REPO_DIR / 'analyze_patch_statistics.py'),
        '--ckpt_dir', str(WORK_CKPT_DIR),
        '--image_root', str(DSE_IMAGE_ROOT),
        '--out', str(RUN_OUTPUT_ROOT / 'patch_attention_dse_all_checkpoints'),
        '--arch', 'vit_small',
        '--patch_size', '16',
        '--checkpoint_key', CHECKPOINT_KEY,
        '--num_metric_images', str(NUM_DSE_IMAGES),
        '--num_vis_images', str(NUM_VIS_IMAGES),
        '--batch_size', str(PATCH_DIAG_BATCH_SIZE),
        '--num_workers', str(PATCH_DIAG_NUM_WORKERS),
        '--max_spectrum_tokens', str(MAX_SPECTRUM_TOKENS),
        '--max_kmeans_tokens', str(MAX_KMEANS_TOKENS),
        '--dse_group_stride', str(PATCH_DSE_GROUP_STRIDE),
        '--seed', '0',
    ]
    if STRICT_INTERNAL_EPOCH:
        cmd.append('--strict_internal_epoch')

    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping raw/L2 patch diagnostics because RUN_PATCH_DIAGNOSTICS=False')

## 6. Optional plots and Markdown report

In [ ]:
import subprocess
import sys

summary_csv = RUN_OUTPUT_ROOT / 'patch_attention_dse_all_checkpoints' / 'patch_attention_dse_summary.csv'
fig_dir = RUN_OUTPUT_ROOT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

if RUN_PLOT_REPORT and summary_csv.is_file():
    plot_cmd = [
        sys.executable,
        str(REPO_DIR / 'plot_dense_diagnostics.py'),
        '--summary_csv', str(summary_csv),
        '--out_dir', str(fig_dir),
    ]
    if VOC_JSON_FOR_REPORT is not None:
        plot_cmd.extend(['--voc_json', str(VOC_JSON_FOR_REPORT)])
    print(' '.join(plot_cmd))
    subprocess.run(plot_cmd, check=True)

    report_cmd = [
        sys.executable,
        str(REPO_DIR / 'make_summary_report.py'),
        '--summary_csv', str(summary_csv),
        '--out', str(RUN_OUTPUT_ROOT / 'summary_report.md'),
    ]
    if VOC_JSON_FOR_REPORT is not None:
        report_cmd.extend(['--voc_json', str(VOC_JSON_FOR_REPORT)])
    print(' '.join(report_cmd))
    subprocess.run(report_cmd, check=True)
elif RUN_PLOT_REPORT:
    print('No patch summary CSV found; skipping combined raw/L2 plots and summary report:', summary_csv)
    if VOC_JSON_FOR_REPORT is not None:
        print('VOC results are still available at:', VOC_JSON_FOR_REPORT)
else:
    print('Skipping plots/report because RUN_PLOT_REPORT=False')

## 7. Plot combined curves and write the run report


In [ ]:
import json
import pandas as pd

if INSPECT_OUTPUTS:
    print('RUN_OUTPUT_ROOT:', RUN_OUTPUT_ROOT)
    print('summary exists:', summary_csv.is_file(), summary_csv)
    print('VOC json exists:', VOC_JSON_FOR_REPORT is not None and Path(VOC_JSON_FOR_REPORT).is_file(), VOC_JSON_FOR_REPORT)
    print('report exists:', (RUN_OUTPUT_ROOT / 'summary_report.md').is_file())

    for name in [
        'fig_dense_diagnostics_summary.png',
        'fig_raw_vs_l2_dse.png',
        'fig_raw_vs_l2_class_sep.png',
        'fig_raw_vs_l2_spectrum.png',
        'combined_dense_summary.csv',
    ]:
        path = fig_dir / name
        print(name, path.is_file(), path)

    if summary_csv.is_file():
        df = pd.read_csv(summary_csv)
        required = [
            'raw_dse',
            'l2_dse',
            'raw_class_sep_avg',
            'l2_class_sep_avg',
            'raw_effective_rank',
            'l2_effective_rank',
            'raw_top1_eigen_ratio',
            'l2_top1_eigen_ratio',
            'patch_norm_mean',
            'patch_norm_p90',
        ]
        for col in required:
            print(col, col in df.columns)

        display_cols = ['epoch'] + [col for col in required if col in df.columns]
        display(df[display_cols])
    else:
        print('No patch diagnostic summary CSV to display.')

    if VOC_JSON_FOR_REPORT is not None and Path(VOC_JSON_FOR_REPORT).is_file():
        voc_results = json.loads(Path(VOC_JSON_FOR_REPORT).read_text())
        print('\n=== VOC results preview ===')
        print(json.dumps(voc_results[:5], indent=2))
        if len(voc_results) > 5:
            print('... last:', json.dumps(voc_results[-1], indent=2))

    report_path = RUN_OUTPUT_ROOT / 'summary_report.md'
    if report_path.is_file():
        print('\n=== summary_report.md ===')
        print(report_path.read_text()[:5000])
else:
    print('Skipping output inspection because INSPECT_OUTPUTS=False')

## 8. Inspect outputs

All persistent raw/L2 validation outputs are saved under `RUN_OUTPUT_ROOT`, for example `/content/drive/MyDrive/dino_dense_degradation_eval/to_epoch_0215_raw_l2/`. VOC is skipped by default and read from the matching base run, for example `/content/drive/MyDrive/dino_dense_degradation_eval/to_epoch_0215/voc_all_checkpoints/voc_miou_results.json`.


In [ ]:
import tarfile

archive_path = Path('/content') / f'{RUN_OUTPUT_ROOT.name}.tar.gz'
if CREATE_OUTPUT_ARCHIVE:
    if archive_path.exists():
        archive_path.unlink()
    with tarfile.open(archive_path, 'w:gz') as tar:
        tar.add(RUN_OUTPUT_ROOT, arcname=RUN_OUTPUT_ROOT.name)
    print('archive:', archive_path, archive_path.stat().st_size / 1024 / 1024, 'MB')
else:
    print('Skipping archive because CREATE_OUTPUT_ARCHIVE=False')